---
---
# **Tutorial 3A:** *Model Context Protocol (MCP)*
### *From hand-written custom integrations to a standard protocol*
---
---

### QUESTIONS FOR TODAY
> 1. *In Tutorial 5 we already made tool calling work. So what problem is MCP actually solving?*
> 2. *If I write a tool today, how does some other application - one I have never seen - use it tomorrow?*

### TODAY'S SCENARIO
> A student types: *"I want to take CS430. Am I eligible, and are there seats?"*

> We will build a **Campus Course Advisor** - first the way we did it in Tutorial 5 (by hand), then the same thing through MCP, so you can see exactly what changes.

### OUR ROADMAP
| Step | What we do |
|---|---|
| 1 | **The manual way** - rebuild Tutorial 5's integration code, and count what it costs us |
| 2 | **The MCP way** - the same tools, with the integration code deleted |
| 3 | **Compare** - hand-written schema vs auto-generated, side by side |
| 4 | **Call tools through MCP** - `list_tools` and `call_tool` |
| 5 | **Connect MCP to our local model** - the full loop |
| 6 | **Why a standard matters** - one server, any host |

### THE ONE IDEA
> **MCP does not change what a tool is. It standardises how tools are described, discovered and called - so anyone's tool works in anyone's app.**

> *Everything runs locally today. No API keys, no rate limits.*

---
# **Step 0: Setting Up**
---

We need Ollama running locally and the `mcp` package.

In [ ]:
# ---------------------------------------------------------
# 1. INSTALL THE MCP SDK
# ---------------------------------------------------------
%pip install -q mcp

# ---------------------------------------------------------
# 2. BRING IN OUR TOOLKITS
# ---------------------------------------------------------
import json                                     # to look at schemas
import csv                                      # to read our catalog
import requests                                 # to talk to Ollama
from mcp.server.mcpserver import MCPServer      # MCP 2.x

# ---------------------------------------------------------
# 3. POINT AT THE LOCAL MODEL
# ---------------------------------------------------------
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen3.5:4b"      # 3.4 GB, supports tool calling

# ---------------------------------------------------------
# 4. LOAD THE COURSE CATALOG
# ---------------------------------------------------------
courses = {}
with open("data/course_catalog.csv") as f:
    for row in csv.DictReader(f):
        courses[row["course_code"]] = row

print("Courses loaded:", len(courses))
print("Codes:", ", ".join(list(courses)[:8]), "...")

In [ ]:
# ---------------------------------------------------------
# A LOOK AT ONE COURSE
# ---------------------------------------------------------
for key, value in courses["CS430"].items():
    print(f"  {key:15s}: {value}")

---
# **Step 1: The Manual Way (a quick rebuild of Tutorial 5)**
---

Let's write our three tools the way we did in Tutorial 5. The functions themselves are the easy part - watch what we have to write *around* them.

In [ ]:
# ---------------------------------------------------------
# THE THREE TOOL FUNCTIONS (ordinary Python - the easy part)
# ---------------------------------------------------------
def get_course_info(course_code: str) -> str:
    """Get the title, department, credits and instructor for a course code."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    return (f"[{c['course_code']}] {c['title']} ({c['department']}), "
            f"{c['credits']} credits, taught by {c['instructor']}")


def check_seats(course_code: str) -> str:
    """Check how many seats are still available in a course."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    left = int(c["seats_total"]) - int(c["seats_taken"])
    if left == 0:
        return f"[{c['course_code']}] {c['title']} is FULL (0 of {c['seats_total']} seats left)."
    return f"[{c['course_code']}] {c['title']} has {left} of {c['seats_total']} seats left."


def get_prerequisites(course_code: str) -> str:
    """Get the prerequisite courses that must be completed before a course."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    if c["prerequisites"] == "NONE":
        return f"[{c['course_code']}] {c['title']} has no prerequisites."
    return f"[{c['course_code']}] {c['title']} requires: {c['prerequisites'].replace(';', ', ')}"


print(check_seats("CS220"))
print(get_prerequisites("CS430"))

In [ ]:
# ---------------------------------------------------------
# NOW THE INTEGRATION WE HAVE TO HAND-WRITE (the tedious part)
# ---------------------------------------------------------

# (a) A JSON schema for EVERY tool - by hand, every time
MANUAL_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_course_info",
            "description": "Get the title, department, credits and instructor for a course code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_code": {"type": "string", "description": "Course code, e.g. CS430"},
                },
                "required": ["course_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_seats",
            "description": "Check how many seats are still available in a course.",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_code": {"type": "string", "description": "Course code, e.g. CS430"},
                },
                "required": ["course_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_prerequisites",
            "description": "Get the prerequisite courses that must be completed before a course.",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_code": {"type": "string", "description": "Course code, e.g. CS430"},
                },
                "required": ["course_code"],
            },
        },
    },
]

# (b) A registry mapping names to functions - by hand
MANUAL_REGISTRY = {                         # Registry Name (key-leftside) : Function Name (value-rightside)
    "get_course_info":   get_course_info,
    "check_seats":       check_seats,
    "get_prerequisites": get_prerequisites,
}

print("Hand-written schema lines:", len(json.dumps(MANUAL_TOOLS, indent=2).split("\n")))
print("Tools described          :", len(MANUAL_TOOLS))

Look at the Count!!

~60 lines of JSON schema for **three tiny functions**.

Now imagine **thirty** tools. *Writing out tool definitions by hand takes a massive amount of repetitive code.*

> 🚨 The real problem is not that this is tedious. It is that **the description of the tool lives separately from the tool itself.**

*In the manual approach, you have to write your actual Python function in one place, and then separately write a bulky JSON description (its name, parameters..) somewhere else in your code.*

> The **Danger**: If you update your Python function later (e.g., changing an argument name), it's very easy to forget to update the separate JSON description. When they get out of sync, your code breaks silently or the model gets confused.

That is exactly the problem MCP solves.

---
# **Step 2: The MCP Way**
---

Same three functions. But now we simply **decorate** them, and MCP reads the function itself - its name, its type hints, its docstring - to build the schema for us.

In [ ]:
# ---------------------------------------------------------
# CREATE AN MCP SERVER
# ---------------------------------------------------------
# "Server" sounds heavy, but it is just an object that holds tools
# and knows how to describe and run them in a standard way.
campus = MCPServer("Campus Advisor")


# ---------------------------------------------------------
# THE SAME TOOLS - now with a decorator, and NO JSON
# ---------------------------------------------------------
@campus.tool()
def get_course_info(course_code: str) -> str:
    """Get the title, department, credits and instructor for a course code."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    return (f"[{c['course_code']}] {c['title']} ({c['department']}), "
            f"{c['credits']} credits, taught by {c['instructor']}")


@campus.tool()
def check_seats(course_code: str) -> str:
    """Check how many seats are still available in a course."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    left = int(c["seats_total"]) - int(c["seats_taken"])
    if left == 0:
        return f"[{c['course_code']}] {c['title']} is FULL (0 of {c['seats_total']} seats left)."
    return f"[{c['course_code']}] {c['title']} has {left} of {c['seats_total']} seats left."


@campus.tool()
def get_prerequisites(course_code: str) -> str:
    """Get the prerequisite courses that must be completed before a course."""
    c = courses.get(course_code.upper().strip())
    if not c:
        return f"{course_code} is not in the catalog."
    if c["prerequisites"] == "NONE":
        return f"[{c['course_code']}] {c['title']} has no prerequisites."
    return f"[{c['course_code']}] {c['title']} requires: {c['prerequisites'].replace(';', ', ')}"


print("Tools registered on the MCP server. No JSON written by us.")

Look at what disappeared:

* ❌ no `MANUAL_TOOLS` JSON
* ❌ no `MANUAL_REGISTRY` dictionary
* ✅ just `@campus.tool()` on top of functions we were writing anyway

The **docstring is now the description** the model reads, and the **type hints are now the parameter schema.** The description of the tool now lives *with* the tool - they can no longer drift apart.

---
# **Step 3: What MCP Generated For Us**
---

Let's ask the server to describe its own tools. In MCP this is called `list_tools()` - and it is how *any* client discovers what a server can do.

In [ ]:
# ---------------------------------------------------------
# ASK THE SERVER WHAT IT CAN DO
# ---------------------------------------------------------
# Note the 'await' - MCP is asynchronous. In a notebook you can
# use await directly at the top level like this.
tools = await campus.list_tools()

print("=== MCP AUTO-GENERATED SCHEMAS ===\n")
for t in tools:
    print(f"Tool        : {t.name}")
    print(f"Description : {t.description}")           # <- from the docstring
    print(f"Input schema:")
    print(json.dumps(t.input_schema, indent=4))       # <- from the type hints
    print()

Compare that with the JSON we hand-wrote in Step 1. **It is the same information** - name, description, parameter types, required fields - except we did not type a single character of it.

> 💡 One nice detail: MCP inferred `"type": "string"` purely from the annotation `course_code: str`. If you had written `credits: int`, it would say `integer`. Your type hints stopped being documentation and became a contract.

### ❓ Quick Poll
> If you rename the argument `course_code` to `subject_code` in the function:
> **(a)** the manual version breaks silently · **(b)** the MCP version updates itself · **(c)** both

<details><summary><b>Reveal</b></summary>

**Both (a) and (b) are true** - that is the whole point. The hand-written JSON still says `course_code`, so the model keeps sending an argument your function no longer accepts, and you get a `TypeError` at runtime. The MCP version regenerates the schema from the function signature, so it simply stays correct.
</details>

---
# **Step 4: Calling Tools Through MCP**
---

The second half of the protocol is `call_tool(name, arguments)` - the standard way any client asks any server to run something.

In [ ]:
# ---------------------------------------------------------
# A SMALL HELPER TO READ MCP RESULTS
# ---------------------------------------------------------
# MCP returns a structured result object rather than a bare value,
# because a tool can return text, images, or several items.
# This helper just pulls out the plain Python value.
def read_result(result):
    """Turn an MCP CallToolResult into a plain Python value."""
    if result.structured_content:                    # simple values arrive wrapped
        return result.structured_content.get("result", result.structured_content)
    texts = [block.text for block in result.content]
    return texts[0] if len(texts) == 1 else texts


# ---------------------------------------------------------
# CALL TOOLS THE MCP WAY
# ---------------------------------------------------------
r1 = await campus.call_tool("get_course_info", {"course_code": "EN220"})
print(read_result(r1))

r2 = await campus.call_tool("check_seats", {"course_code": "MA201"})
print(read_result(r2))

r3 = await campus.call_tool("get_prerequisites", {"course_code": "CS430"})
print(read_result(r3))

# and a course that does not exist
r4 = await campus.call_tool("check_seats", {"course_code": "PH404"})
print(read_result(r4))

Notice we never touched a registry. **We asked the server by name, and it dispatched to the right function itself.** That dispatcher we hand-wrote in Tutorial 5 is now part of the protocol.

---
# **Step 5: Connecting MCP to Our Local Model**
---

Our local model speaks Ollama's tool format, not MCP's. So we do one small translation - and crucially, **we build the model's tool list from the MCP server itself**, instead of maintaining a second copy by hand.

In [ ]:
# ---------------------------------------------------------
# TRANSLATE MCP SCHEMAS -> OLLAMA'S TOOL FORMAT
# ---------------------------------------------------------
# This is derived from the server, so it can never fall out of sync.
mcp_tools = await campus.list_tools()

ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.input_schema,
        },
    }
    for t in mcp_tools
]

print("Tools handed to the model:", [t["function"]["name"] for t in ollama_tools])

In [ ]:
# ---------------------------------------------------------
# THE FULL LOOP: ASK -> MODEL REQUESTS -> MCP RUNS -> MODEL ANSWERS
# ---------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a campus course advisor. Always use the tools to look up course "
    "information. Never guess about seats, prerequisites or course details."
)


async def ask(question):
    """Same 4-step handshake as Tutorial 5 - but MCP does the dispatching."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    # --- Step 1: ask, with the tool menu (built from MCP) ---
    r = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME, "messages": messages,
        "tools": ollama_tools, "stream": False,
        "think": False,          # keeps the local model fast
    }).json()

    message = r["message"]

    # --- If no tool was wanted, it already answered ---
    if "tool_calls" not in message:
        print("----------(No tool needed)")
        return message.get("content", "")

    # --- Step 2: read the request ---
    fc = message["tool_calls"][0]["function"]
    print(f"----------Model requested : {fc['name']}({fc['arguments']})")

    # --- Step 3: DISPATCH THROUGH MCP (no registry of ours!) ---
    mcp_result = await campus.call_tool(fc["name"], fc["arguments"])
    result = read_result(mcp_result)
    print(f"----------MCP returned    : {result}")

    # --- Step 4: send the result back ---
    messages += [message, {"role": "tool", "content": str(result)}]

    final = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME, "messages": messages,
        "tools": ollama_tools, "stream": False, "think": False,
    }).json()

    return final["message"]["content"]

In [ ]:
# ---------------------------------------------------------
# TEST 1: seats
# ---------------------------------------------------------
print(await ask("Are there any seats left in MA201?"))

In [ ]:
# ---------------------------------------------------------
# TEST 2: prerequisites (same code, different tool chosen)
# ---------------------------------------------------------
print(await ask("What do I need to complete before taking CS430?"))

Same four-step handshake you learned in Tutorial 5. The difference is **Step 3**: we no longer look anything up in a registry we maintain. We hand the name straight to MCP and it does the dispatch.

| | Tutorial 5 (manual) | Today (MCP) |
|---|---|---|
| Describe a tool | write JSON by hand | `@server.tool()` decorator |
| Keep a registry | `TOOL_REGISTRY = {...}` | the server *is* the registry |
| Dispatch a call | `TOOL_REGISTRY[name](**args)` | `await server.call_tool(name, args)` |
| Discover tools | you already know them | `await server.list_tools()` |
| Another app can use it | copy-paste your code | connect to the server |

---
# **Step 6: Why a Standard Matters**
---

**The main reason why MCP exits is that last row of the table [Interoperability].** Because MCP creates a universal standard, your tool server doesn't have to be trapped inside the specific Python script where you wrote it.

```
                    ┌──────────────────────────┐
   Claude Desktop ──┤                          │
   VS Code        ──┤   YOUR MCP SERVER        ├── your data / APIs
   Your agent     ──┤   (the tools we wrote)   │
   Someone else's ──┤                          │
   app              └──────────────────────────┘
```

You did not write any integration code for those hosts. **They already know how to speak MCP**, so they can ask your server what it can do (`list_tools`) and then use it (`call_tool`).

> ### 🌐 There is a public registry of these
> [registry.modelcontextprotocol.io](https://registry.modelcontextprotocol.io/) lists MCP servers people have published - for databases, file systems, issue trackers, and so on. Connecting an agent to Postgres becomes "point it at the Postgres MCP server" instead of "write another integration".

### One honest note about how we ran things today

We skipped

```python
campus.run(transport="stdio")     # how a real MCP server is launched
```
in this notebook to keep everything in one file and prevent your kernel from freezing.
You will use this exact command to launch your server as a background process when connecting it to external apps like Claude Desktop.

---
# **What We Built Today**
---

We started with a hand-written JSON, a registry, a dispatcher - and watched MCP absorb all three.

**Three things to carry forward:**

1. **MCP did not change what a tool is.** It is still an ordinary Python function. It changed how tools are *described, discovered and called.*
2. **The schema now comes from your code.** Docstrings and type hints are no longer documentation - they are the interface the model reads.
3. **A standard turns your tools into a service.** Any MCP-capable host can use them without you writing integration code.

And remember what is still true from Tutorial 5: **the model requests, your server executes.** MCP standardises the conversation - it does not hand the model control.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 August 29, Saturday*